# VGG-16 Fine-Tuning (Step 4) — runs on Kaggle with GPU

Fine-tunes a pretrained VGG-16 on `train/`, then predicts on the **same** `test/` split the
Claude agent used, and exports `vgg16_predictions.csv` for the Step 5 comparison.

**Before running:** Settings panel (right) → Accelerator → **GPU**.

**Data:** zip your local `data/splits/` folder and upload it as a Kaggle Dataset
(*+ New Dataset* → upload the zip). It mounts read-only under `/kaggle/input/<your-dataset-name>/`.
Set `DATA_ROOT` in Cell 2 to the folder that contains `train/` and `test/`.

Using the same zipped split (rather than re-downloading Intel inside Kaggle) is what
guarantees the agent and VGG-16 are scored on identical test images.

## Cell 1 — setup & GPU check

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))
# If the GPU list is empty, stop: turn on the GPU accelerator in Settings before continuing.

## Cell 2 — load data

`test_generator` uses `shuffle=False` so `filenames`, `classes`, and `predict()` rows stay aligned —
essential for mapping predictions back to the right file in Cell 5.

In [ ]:
import os

# EDIT THIS to the folder that contains train/ and test/ (inspect /kaggle/input first).
DATA_ROOT = "/kaggle/input/tw-project-splits/splits"

TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR = os.path.join(DATA_ROOT, "test")
IMG_SIZE = (224, 224)   # VGG-16's native input size
BATCH = 32

assert os.path.isdir(TRAIN_DIR), f"Not found: {TRAIN_DIR} — check DATA_ROOT against /kaggle/input"
assert os.path.isdir(TEST_DIR), f"Not found: {TEST_DIR}"

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="categorical", shuffle=True,
)
test_generator = test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH,
    class_mode="categorical", shuffle=False,   # keep order fixed for prediction export
)

NUM_CLASSES = train_generator.num_classes
print("class_indices:", train_generator.class_indices)
print("train samples:", train_generator.samples, "| test samples:", test_generator.samples)
# Confirm all 6 classes appear with sane counts before training.

## Cell 3 — build the model (transfer learning: freeze VGG-16 base)

In [ ]:
base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
base.trainable = False   # freeze pretrained conv layers

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

## Cell 4 — train

Smoke test first: set `EPOCHS = 2` and confirm loss decreases and accuracy climbs above
chance (~16.7% for 6 classes) but isn't pinned at 100% (which would suggest a train/test leak).
Then rerun with `EPOCHS = 15`.

In [ ]:
EPOCHS = 15   # start at 2 for the smoke test, then 15

history = model.fit(
    train_generator,
    validation_data=test_generator,
    epochs=EPOCHS,
)

## Cell 5 — predict on the test set and export

Writes `vgg16_predictions.csv` with the **same columns** the agent CSV uses
(`filepath, true_label, predicted_label`) so Step 5 can align them directly.
`filepath` is stored as `test/<class>/<filename>` (a basename-style relative path) so it
matches across machines regardless of the absolute path on Kaggle vs. local.

In [ ]:
import numpy as np
import pandas as pd

# index -> class name (invert class_indices)
idx_to_class = {v: k for k, v in test_generator.class_indices.items()}

probs = model.predict(test_generator)
pred_idx = probs.argmax(axis=1)

rows = []
for fname, true_i, pred_i in zip(test_generator.filenames, test_generator.classes, pred_idx):
    # fname is '<class>/<file>'; store as 'test/<class>/<file>' for a stable cross-machine key
    rows.append({
        "filepath": "test/" + fname.replace(os.sep, "/"),
        "true_label": idx_to_class[true_i],
        "predicted_label": idx_to_class[pred_i],
    })

df = pd.DataFrame(rows, columns=["filepath", "true_label", "predicted_label"])
out_path = "/kaggle/working/vgg16_predictions.csv"
df.to_csv(out_path, index=False)

acc = (df.true_label == df.predicted_label).mean()
print(f"Wrote {len(df)} predictions to {out_path}")
print(f"Test accuracy: {acc:.3f}")
df.head()

After this runs, download `vgg16_predictions.csv` from the Kaggle output panel
(right side, *Output*) and drop it into your local `results/` folder for Step 5.